# Instructions
File structure
- src: n=100 full patient cases
- out
   - yellow: yellow pages of cases with yellow pages and no red pages  
   - red: red pages of cases with red pages
   - anon: yellow pages anonymized

In [ ]:
%matplotlib inline
!pip install opencv-python
!pip install PyMuPDF
!pip install matplotlib
!pip install pypdf
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np
import fitz
from pypdf import PdfReader, PdfWriter

data_dir = "src_mini"
out_dir = "out"


filenames_full = os.listdir(data_dir)

filenames = []
filenames_removed = []
for fn in filenames_full:
    if fn.endswith(".pdf"):
        filenames.append(fn)
    else:
        filenames_removed.append(fn)

print(filenames)
print(filenames_removed)

# Extract Yellow Sheets

In [ ]:
# Record average page colors (int, RGB)
colormaps = np.ones((len(filenames), 200, 3), dtype=np.uint8)

for idx, fn in enumerate(filenames):
    # Open
    path = os.path.join(data_dir, fn)
    doc = fitz.open(path)
    
    for i, page_idx in enumerate(doc):
        pix = page_idx.get_pixmap(dpi=150)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n) # HWC
        avg = np.mean(img, axis=(0, 1))
        colormaps[idx, i] = avg
    
    print(f"{path} done")

In [ ]:
plt.figure(figsize=(15, 2))
plt.imshow(colormaps, interpolation='nearest')
plt.title("Page average colors")
plt.show()

Color rules:
- Ignore yellow if >4 in a row, at the back end
- Remove document if red
- Red: 320-39, Yellow: 40-80

In [ ]:
# Clear directories
for file in os.listdir(os.path.join(out_dir, 'red')):
    os.remove(os.path.join(out_dir, 'red', file))
for file in os.listdir(os.path.join(out_dir, 'yellow')):
    os.remove(os.path.join(out_dir, 'yellow', file))
for file in os.listdir(os.path.join(out_dir, 'anon')): 
    os.remove(os.path.join(out_dir, 'anon', file))

# Categorize by color
for file_idx in range(len(filenames)):
    yellow_pages_idx = []
    yellow_prev_start = 0
    red_pages_idx = []
    num_pages = len(colormaps[file_idx])
    
    ### Parse in file
    for page_idx in range(num_pages):
        
        pixel_rgb = np.uint8([[colormaps[file_idx, page_idx]]])
        pixel_hsv = cv2.cvtColor(pixel_rgb, cv2.COLOR_RGB2HSV)[0,0]
        hue, sat, val = pixel_hsv
        
        
        # Red
        if (sat > 20 and val > 50) and (hue > 160 or hue <= 0):
            red_pages_idx.append(page_idx)
            #print(pixel_hsv, file_idx, "red")
        
        # Yellow
        elif (sat > 20 and val > 50) and (16 <= hue < 40):
            yellow_pages_idx.append(page_idx)
            #print(pixel_hsv, file_idx, "yellow")
            
            # if a new chain of yellow, mark it so
            if page_idx-1 not in yellow_pages_idx:
                yellow_prev_start = page_idx
            
        # Neither; check if previous was yellow or red
        else: 
            # If previous was yellow, check if > 4 in a row at the back end -> remove those
            if yellow_pages_idx and len(yellow_pages_idx) > 4 and page_idx > num_pages/2:
                del yellow_pages_idx[yellow_pages_idx.index(yellow_prev_start):]
            
            # If previous was red, end loop because we don't need to check more; 
            if red_pages_idx:
                break
    
    ### Save pages
    reader = PdfReader(os.path.join(data_dir, filenames[file_idx]))
    writer = PdfWriter()
    
    # If had red pages, add the red pages to a certain document
    if red_pages_idx:
        for page_idx in red_pages_idx:
            writer.add_page(reader.pages[page_idx])
        with open(os.path.join(out_dir, "red", f"{file_idx} pg{red_pages_idx[0]+1}{('-'+str(red_pages_idx[-1]+1))if len(red_pages_idx)>1 else ''}.pdf"), "wb") as f:
            writer.write(f)

    # No red pages, then add all valid yellow pages to a certain document
    else:
        if yellow_pages_idx: # some might not have valid yellow pages
            for page_idx in yellow_pages_idx:
                writer.add_page(reader.pages[page_idx])
            with open(os.path.join(out_dir, "yellow", f"{file_idx} pg{yellow_pages_idx[0]+1}{('-'+str(yellow_pages_idx[-1]+1))if len(yellow_pages_idx)>1 else ''}.pdf"), "wb") as f:
                writer.write(f)

# Anonymize 
- Only first page has information needed to be anonymized
- Patient name and DOB (age is fine)

In [ ]:
# Static method (not very good)

yellow_fns = os.listdir(os.path.join(out_dir, 'yellow'))
yellow_fns.remove('.gitignore')

for idx, fn in enumerate(yellow_fns):
    # Open
    path = os.path.join(out_dir, 'yellow', fn)
    doc = fitz.open(path)
    page = doc[0]
    height, width = page.rect.height, page.rect.width

    rect1 = fitz.Rect(width*.13, height*.045, width*.54, height*.06)
    rect2 = fitz.Rect(width*.77, height*.045, width*.95, height*.06)
    page.add_redact_annot(rect1, fill=(0, 0, 0))  # Black fill
    page.add_redact_annot(rect2, fill=(0, 0, 0))  # Black fill
    page.apply_redactions()

    doc.save(os.path.join(out_dir, "anon", fn))


In [ ]:
# Contour method (not very good)

yellow_fns = os.listdir(os.path.join(out_dir, 'yellow'))
yellow_fns.remove('.gitignore')

for idx, fn in enumerate(yellow_fns):
    # Open
    path = os.path.join(out_dir, 'yellow', fn)
    doc = fitz.open(path)
    page = doc[0]
    pix = page.get_pixmap(dpi=150)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    
    img = cv2.imread(img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5,5), 0), 50, 150)
    
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    rectangles = []
    for cnt in contours:
        epsilon = 0.02 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        if len(approx) == 4 and cv2.isContourConvex(approx):
            area = cv2.contourArea(approx)
            if area > 1000:  # ignore small noise
                rectangles.append(approx)

    # Sort by area or position if needed
    rectangles = sorted(rectangles, key=cv2.contourArea, reverse=True)

    # Draw for debug
    for rect in rectangles:
        cv2.drawContours(img, [rect], -1, (0, 255, 0), 2)


    
    break
